In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 18 consolidation — Separate throughput from how old the answer is

Read `sessions/18_session.md` first. This is an executed reference, not an assessment of you.

**Evidence:** Real reference point table for measured CPU operations; separate deterministic FIFO simulation.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT/'data/raw/iris.csv').is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Start Jupyter in the extracted listening_to_shape_lab folder.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))


## Step 1

Compare two implementations on a point table derived from real reference data.

Before running: write your expected result and name one way this check could fail.

In [3]:
from shape_lab.stereo import reference_sample, point_cloud
from shape_lab.geometry import exp_so3
from time import perf_counter
_, _, disparity, calibration = reference_sample()
X, _ = point_cloud(disparity, calibration, stride=20)
rotation = exp_so3([.1,.2,.3]); translation = np.array([.2,0.,-.1])
def loop_transform():
    return np.array([rotation @ p + translation for p in X])
def array_transform():
    return X @ rotation.T + translation
assert np.allclose(loop_transform(), array_transform())

## Step 2

Measure only the CPU coordinate transformation, with warm-up and repeated runs.

Before running: write your expected result and name one way this check could fail.

In [4]:
loop_transform(); array_transform()
timings = {}
for name, operation in [('loop', loop_transform), ('array', array_transform)]:
    durations = []
    for _ in range(7):
        start = perf_counter(); operation(); durations.append(perf_counter()-start)
    timings[name] = float(np.median(durations))
print('median CPU seconds:', timings)

median CPU seconds: {'loop': 0.0013463119998959883, 'array': 1.8548000070950366e-05}


## Step 3

A separate FIFO simulation demonstrates backlog. It is not relabeled as measured latency.

Before running: write your expected result and name one way this check could fail.

In [5]:
from shape_lab.temporal import fifo_latencies
arrival = np.arange(100)*.02
latency = fifo_latencies(arrival, np.full(100,.03))
assert np.isclose(latency[0], .03)
assert np.isclose(latency[-1], 1.02)
result = {'points': len(X), 'median_cpu_transform_seconds': timings,
          'timing_boundary': 'in-memory CPU transform only',
          'simulated_first_latency_s': float(latency[0]), 'simulated_last_latency_s': float(latency[-1]),
          'end_to_end_camera_latency_measured': False}

## Keep execution evidence separate from learning evidence

The following saves this reference calculation. Write your own explanation in `my_work/`; no learner score is assigned.

In [6]:
result['stage'] = 18
result['execution_role'] = 'reference_consolidation'
result['data_role'] = 'Real reference point table for measured CPU operations; separate deterministic FIFO simulation.'
output = ROOT/'reports/consolidation_v4/18.json'
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(result, indent=2, allow_nan=False) + '\n')
print('Saved reference result:', output.relative_to(ROOT))


Saved reference result: reports/consolidation_v4/18.json
